# 🇮🇳 Indian Tech Jobs 2025 | EDA + ML

**Author:** Shreyash ([shree0910](https://www.kaggle.com/shree0910))
**Dataset:** Indian Tech Job Market 2025 — Scraped from Naukri.com
**Date:** June 2025

---

## 📋 What This Notebook Covers

| # | Section | What You'll Learn |
|---|---------|-------------------|
| 1 | Setup & Data Loading | Import libraries, load dataset, first look |
| 2 | Exploratory Data Analysis | Roles, cities, salary, work mode, skills |
| 3 | Data Preprocessing | Feature selection, encoding, train/test split |
| 4 | ML Task 1 — Work Mode Classifier | Predict Remote / Hybrid / On-site (90%+ accuracy) |
| 5 | ML Task 2 — Salary Prediction | Predict salary in LPA (regression) |
| 6 | Feature Importance | Which features drive predictions the most |
| 7 | Conclusion | Key findings & takeaways |

> 💡 **Beginner-friendly:** Every step is explained in plain English before the code.
> No prior ML knowledge needed — just follow along!


## 📦 Section 1 — Setup & Data Loading

We start by importing **libraries** — think of them as toolboxes that give us pre-built tools so we don't have to write everything from scratch.

In [ ]:
# ── Standard Data Libraries ──────────────────────────────────
import pandas as pd           # for tables (DataFrames) — like Excel in Python
import numpy as np            # for numbers and math
import matplotlib.pyplot as plt  # for basic charts
import seaborn as sns         # for beautiful statistical charts
import warnings
warnings.filterwarnings('ignore')  # hide unimportant warnings

# ── Machine Learning Libraries ────────────────────────────────
from sklearn.model_selection  import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.ensemble         import (RandomForestClassifier, RandomForestRegressor,
                                      GradientBoostingRegressor)
from sklearn.linear_model     import LogisticRegression, Ridge
from sklearn.metrics          import (accuracy_score, classification_report,
                                      confusion_matrix, r2_score,
                                      mean_absolute_error, mean_squared_error)
from sklearn.pipeline         import Pipeline
from sklearn.compose          import ColumnTransformer
from sklearn.preprocessing    import OneHotEncoder
from sklearn.impute            import SimpleImputer

try:
    from xgboost import XGBClassifier, XGBRegressor
    XGBOOST = True
    print("✅ XGBoost available")
except ImportError:
    XGBOOST = False
    print("⚠ XGBoost not found — install with: pip install xgboost")

# ── Plot Style ───────────────────────────────────────────────
plt.rcParams['figure.dpi']      = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
PALETTE = ['#2E86AB','#A23B72','#F18F01','#C73E1D','#44BBA4',
           '#393E41','#E94F37','#7B68EE','#F5A623','#5C6BC0']
sns.set_palette(PALETTE)

print("✅ All libraries imported!")


### 1.1 — Load the Dataset

`pd.read_csv()` reads a CSV file into a DataFrame — a table with rows and columns.

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────
# On Kaggle: /kaggle/input/indian-tech-job-market-2025/indian_tech_jobs_2025_cleaned.csv
df = pd.read_csv('/kaggle/input/indian-tech-job-market-2025/indian_tech_jobs_2025_cleaned.csv')

print(f"✅ Dataset loaded successfully!")
print(f"   Rows    : {len(df):,}")
print(f"   Columns : {df.shape[1]}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")


In [ ]:
# First 5 rows — always look at your data first!
df.head()


In [ ]:
# Data types of each column
print("Data Types:")
print(df.dtypes.to_string())

print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")
print("✅ Zero missing values — dataset is fully clean!")


In [ ]:
# Summary statistics for numeric columns
# count = non-null rows, mean = average, std = spread, 25%/50%/75% = percentiles
df.describe().round(2)


## 📊 Section 2 — Exploratory Data Analysis (EDA)

**EDA = Exploring the data before modelling.**
We want to understand patterns, distributions, and relationships.


### 2.1 — Job Role Distribution
Which tech roles are most in demand across India?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

role_counts = df['role_category'].value_counts()

# Bar chart
axes[0].barh(role_counts.index[::-1], role_counts.values[::-1], color=PALETTE[:6])
axes[0].set_title('Job Postings by Role', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Listings')
for i, v in enumerate(role_counts.values[::-1]):
    axes[0].text(v + 20, i, f'{v:,}', va='center', fontsize=9)

# Pie chart
axes[1].pie(role_counts.values, labels=role_counts.index,
            autopct='%1.1f%%', colors=PALETTE[:6], startangle=140)
axes[1].set_title('Role Share (%)', fontsize=13, fontweight='bold')

plt.suptitle('📋 Tech Role Demand — India 2025', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('01_role_distribution.png', bbox_inches='tight')
plt.show()
print("💡 Insight: Data Scientist dominates with 27.8%, followed by Data Analyst (20.4%).")


### 2.2 — City-wise Distribution
Which Indian cities hire the most tech professionals?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

city_counts = df['primary_city'].value_counts().head(10)

axes[0].bar(city_counts.index, city_counts.values, color=PALETTE)
axes[0].set_title('Top 10 Cities — Job Count', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Listings')
axes[0].tick_params(axis='x', rotation=35)
for i, v in enumerate(city_counts.values):
    axes[0].text(i, v + 30, f'{v:,}', ha='center', fontsize=8)

axes[1].pie(city_counts.values[:6], labels=city_counts.index[:6],
            autopct='%1.1f%%', colors=PALETTE[:6], startangle=140)
axes[1].set_title('Top 6 Cities Share', fontsize=13, fontweight='bold')

plt.suptitle('🏙️ Geographic Distribution of Tech Jobs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('02_city_distribution.png', bbox_inches='tight')
plt.show()
print("💡 Insight: Mumbai leads (14.1%), followed by Bangalore (12.2%) and Chennai (11.5%).")


### 2.3 — Work Mode Trends
Remote, Hybrid, or On-site?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

wm = df['work_mode'].value_counts()

# Pie
axes[0].pie(wm.values, labels=wm.index, autopct='%1.1f%%',
            colors=['#2E86AB','#44BBA4','#F18F01'],
            startangle=140, wedgeprops={'edgecolor':'white','lw':2})
axes[0].set_title('Overall Work Mode', fontsize=12, fontweight='bold')

# By role
wm_role = df.groupby(['role_category','work_mode']).size().unstack(fill_value=0)
wm_role.plot(kind='bar', ax=axes[1], color=['#2E86AB','#44BBA4','#F18F01'])
axes[1].set_title('Work Mode by Role', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Mode', fontsize=8)

# By top 6 cities
top6 = df['primary_city'].value_counts().head(6).index
wm_city = (df[df['primary_city'].isin(top6)]
           .groupby(['primary_city','work_mode']).size().unstack(fill_value=0))
wm_city.plot(kind='bar', ax=axes[2], color=['#2E86AB','#44BBA4','#F18F01'])
axes[2].set_title('Work Mode by City', fontsize=12, fontweight='bold')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=35)
axes[2].legend(title='Mode', fontsize=8)

plt.suptitle('🏠 Remote vs Hybrid vs On-site — 2025', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('03_work_mode.png', bbox_inches='tight')
plt.show()
print(f"💡 Insight: {wm['On-site']/len(df)*100:.1f}% On-site | {wm['Remote']/len(df)*100:.1f}% Remote | {wm['Hybrid']/len(df)*100:.1f}% Hybrid")


### 2.4 — Salary Analysis
Salary is only disclosed for ~12% of listings. Let's analyze those.

In [ ]:
sal_df = df[df['salary_disclosed'] == True].copy()
sal_df = sal_df[sal_df['salary_midpoint_lpa'].between(1, 80)]  # remove outliers

print(f"Salary disclosed: {len(sal_df):,} jobs ({len(sal_df)/len(df)*100:.1f}%)")
print(f"Salary range    : ₹{sal_df['salary_midpoint_lpa'].min():.1f} – ₹{sal_df['salary_midpoint_lpa'].max():.1f} LPA")
print(f"Median salary   : ₹{sal_df['salary_midpoint_lpa'].median():.1f} LPA")
print(f"Mean salary     : ₹{sal_df['salary_midpoint_lpa'].mean():.1f} LPA")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(sal_df['salary_midpoint_lpa'], bins=35, color='#2E86AB',
             edgecolor='white', alpha=0.85)
med = sal_df['salary_midpoint_lpa'].median()
axes[0].axvline(med, color='red', linestyle='--', lw=2,
                label=f'Median: ₹{med:.1f} LPA')
axes[0].set_title('Salary Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Salary (LPA)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot by role
order = (sal_df.groupby('role_category')['salary_midpoint_lpa']
               .median().sort_values(ascending=False).index)
sns.boxplot(data=sal_df, x='salary_midpoint_lpa', y='role_category',
            order=order, palette=PALETTE[:6], ax=axes[1])
axes[1].set_title('Salary by Role (Box Plot)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Salary Midpoint (LPA)')
axes[1].set_ylabel('')

plt.suptitle('💰 Salary Analysis — Indian Tech Jobs 2025', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('04_salary_analysis.png', bbox_inches='tight')
plt.show()
print("💡 MLE and Data Scientists earn the highest median salaries.")


### 2.5 — Experience vs Salary
Does more experience always mean more pay?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_df = sal_df.dropna(subset=['experience_min_yrs'])

ax.scatter(plot_df['experience_min_yrs'], plot_df['salary_midpoint_lpa'],
           c=pd.Categorical(plot_df['role_category']).codes,
           alpha=0.4, s=20, cmap='tab10')

# Trend line (linear regression)
z = np.polyfit(plot_df['experience_min_yrs'], plot_df['salary_midpoint_lpa'], 1)
p = np.poly1d(z)
x_range = np.linspace(0, plot_df['experience_min_yrs'].max(), 100)
ax.plot(x_range, p(x_range), 'r--', lw=2.5, label=f'Trend (slope ≈ ₹{z[0]:.1f} LPA/yr)')

ax.set_title('Experience vs Salary', fontsize=13, fontweight='bold')
ax.set_xlabel('Min Experience (Years)')
ax.set_ylabel('Salary Midpoint (LPA)')
ax.legend()
plt.tight_layout()
plt.savefig('05_exp_vs_salary.png', bbox_inches='tight')
plt.show()
print(f"💡 Each extra year of experience adds ~₹{z[0]:.1f} LPA on average.")


### 2.6 — Skill Domain & Salary Tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sd = df['skill_domain'].value_counts()
axes[0].barh(sd.index[::-1], sd.values[::-1], color=PALETTE[:len(sd)])
axes[0].set_title('Skill Domain Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Count')

st_order = ['Entry (<3 LPA)','Junior (3-6 LPA)','Mid (6-10 LPA)',
            'Senior (10-20 LPA)','Leadership (20+ LPA)','Undisclosed']
st = df['salary_tier'].value_counts()
st = st.reindex([s for s in st_order if s in st.index])
bars = axes[1].bar(range(len(st)), st.values, color=PALETTE[:len(st)])
axes[1].set_xticks(range(len(st)))
axes[1].set_xticklabels(st.index, rotation=25, ha='right', fontsize=9)
axes[1].set_title('Salary Tier Distribution', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                f'{int(bar.get_height()):,}', ha='center', fontsize=8)

plt.suptitle('🔧 Skills & Salary Tiers Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('06_skills_salary_tier.png', bbox_inches='tight')
plt.show()


### 2.7 — Correlation Heatmap
Which numeric features are related to each other?

In [ ]:
num_cols = ['experience_min_yrs','experience_max_yrs','salary_min_lpa',
            'salary_max_lpa','salary_midpoint_lpa','skills_count',
            'company_rating','days_since_posted']

# Correlation matrix: values close to 1 = strong positive relation
# values close to -1 = strong negative relation
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle (duplicate)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax, linewidths=0.5,
            cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('07_correlation.png', bbox_inches='tight')
plt.show()
print("💡 salary_min & salary_max are highly correlated (expected).")
print("   experience_min shows moderate positive correlation with salary.")


## 🔧 Section 3 — Data Preprocessing

Before feeding data into a model, we must:
1. **Select features** — which columns to use as inputs
2. **Encode categories** — convert text (e.g. "Bangalore") → numbers
3. **Scale numbers** — bring all numbers to a similar range
4. **Split data** — 80% for training, 20% for testing

We use `sklearn Pipeline` which chains these steps automatically.


In [ ]:
# ── Define features for TASK 1 (Work Mode Classifier) ────────

# These are the INPUT features (what the model sees)
FEATURES_CLF = [
    'role_category',        # Data Scientist, MLE, etc.
    'primary_city',         # Bangalore, Mumbai, etc.
    'experience_min_yrs',   # minimum years of experience
    'experience_max_yrs',   # maximum years of experience
    'skills_count',         # how many skills are listed
    'company_rating',       # company rating out of 5
    'company_size_bucket',  # Large / Mid / Small
    'skill_domain',         # AI/ML/DL, BI, Cloud, etc.
    'is_senior',            # 1 if senior role, 0 otherwise
    'is_fresher_friendly',  # 1 if freshers can apply
    'days_since_posted',    # how recently was it posted
]
TARGET_CLF = 'work_mode'   # OUTPUT we want to predict

# Prepare the data
clf_df = df[FEATURES_CLF + [TARGET_CLF]].copy()
clf_df['is_senior']           = clf_df['is_senior'].astype(int)  # bool → 1/0
clf_df['is_fresher_friendly'] = clf_df['is_fresher_friendly'].astype(int)

print(f"✅ Features selected: {len(FEATURES_CLF)}")
print(f"   Dataset size: {len(clf_df):,} rows")
print(f"\nTarget class distribution:")
print(clf_df[TARGET_CLF].value_counts())


In [ ]:
# ── Identify numeric vs categorical columns ───────────────────

# Numeric features: already numbers — just fill missing + scale
num_features_clf = ['experience_min_yrs','experience_max_yrs','skills_count',
                    'company_rating','is_senior','is_fresher_friendly','days_since_posted']

# Categorical features: text labels — need to convert to numbers
cat_features_clf = ['role_category','primary_city','company_size_bucket','skill_domain']

# ── Build Preprocessing Pipeline ──────────────────────────────
# Pipeline automatically applies steps in order
preprocessor_clf = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),   # fill missing with median
        ('scaler',  StandardScaler()),                   # scale to mean=0, std=1
    ]), num_features_clf),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),        # fill missing
        ('ohe',     OneHotEncoder(handle_unknown='ignore',           # text → 0/1 columns
                                  sparse_output=False)),
    ]), cat_features_clf),
])

# ── Encode target labels ───────────────────────────────────────
# LabelEncoder converts: 'Remote'→0, 'Hybrid'→1, 'On-site'→2
le_wm = LabelEncoder()
y_clf = le_wm.fit_transform(clf_df[TARGET_CLF])
X_clf = clf_df.drop(columns=[TARGET_CLF])

print(f"Target classes: {dict(enumerate(le_wm.classes_))}")

# ── Train / Test Split ─────────────────────────────────────────
# 80% for training the model, 20% for testing (held out, never seen)
# stratify=y ensures same class ratio in both splits
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print(f"\n✅ Train set: {len(X_train_clf):,} rows")
print(f"   Test set : {len(X_test_clf):,} rows")


## 🤖 Section 4 — ML Task 1: Work Mode Classifier

**Goal:** Predict whether a job is **Remote**, **Hybrid**, or **On-site**

We compare 3 models:
| Model | How it works (simple) |
|-------|----------------------|
| **Logistic Regression** | Draws boundaries between classes (fast, simple baseline) |
| **Random Forest** | Builds 200 decision trees and takes a majority vote |
| **XGBoost** | Builds trees one after another, each fixing the previous one's mistakes |

We use **5-Fold Cross Validation** — splits data into 5 parts, trains 5 times,
averages the score. More reliable than a single train/test split.


In [ ]:
# ── Define Models ─────────────────────────────────────────────
# class_weight='balanced' handles the imbalance
# (On-site has 18K rows, Remote only 2K — balanced weights compensate)
models_clf = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42,
                                               class_weight='balanced'),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, random_state=42,
                                                   class_weight='balanced', n_jobs=-1),
}
if XGBOOST:
    models_clf['XGBoost'] = XGBClassifier(n_estimators=200, random_state=42,
                                           use_label_encoder=False,
                                           eval_metric='mlogloss',
                                           verbosity=0, n_jobs=-1,
                                           learning_rate=0.05,
                                           max_depth=6, subsample=0.8)

# 5-Fold Stratified Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_clf = []

print("Training models... (this may take 1-2 minutes)\n")

for name, model in models_clf.items():
    # Pipeline: preprocessor + model in one clean step
    pipe = Pipeline([('prep', preprocessor_clf), ('model', model)])
    pipe.fit(X_train_clf, y_train_clf)

    y_pred = pipe.predict(X_test_clf)
    acc    = accuracy_score(y_test_clf, y_pred)
    cv_acc = cross_val_score(pipe, X_clf, y_clf, cv=skf,
                             scoring='accuracy', n_jobs=-1).mean()

    print(f"  {name:<22} Test Acc={acc:.4f}  CV-5fold={cv_acc:.4f}")
    results_clf.append({
        'Model': name, 'Test Accuracy': acc, 'CV Accuracy': cv_acc
    })

clf_results_df = pd.DataFrame(results_clf).sort_values('CV Accuracy', ascending=False)
print(f"\n🏆 Best Model: {clf_results_df.iloc[0]['Model']}")
print(f"   CV Accuracy: {clf_results_df.iloc[0]['CV Accuracy']:.4f} ({clf_results_df.iloc[0]['CV Accuracy']*100:.1f}%)")


In [ ]:
# ── Model Comparison Chart ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(clf_results_df))

bars1 = ax.bar(x - 0.2, clf_results_df['Test Accuracy'], 0.38,
               color=PALETTE[:len(clf_results_df)], label='Test Accuracy')
bars2 = ax.bar(x + 0.2, clf_results_df['CV Accuracy'],  0.38,
               color=PALETTE[:len(clf_results_df)], alpha=0.55, label='CV-5fold Accuracy')

ax.set_xticks(x)
ax.set_xticklabels(clf_results_df['Model'], rotation=10)
ax.set_ylim(0.65, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison — Work Mode Classifier', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(0.9, color='green', linestyle=':', lw=1.5, alpha=0.7, label='90% line')

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f'{bar.get_height():.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('08_model_comparison.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Best Model — Detailed Evaluation ─────────────────────────
best_clf_name  = clf_results_df.iloc[0]['Model']
best_clf_model = models_clf[best_clf_name]
best_clf_pipe  = Pipeline([('prep', preprocessor_clf), ('model', best_clf_model)])
best_clf_pipe.fit(X_train_clf, y_train_clf)
y_pred_best = best_clf_pipe.predict(X_test_clf)

print(f"🏆 Best Model: {best_clf_name}")
print(f"   Test Accuracy : {accuracy_score(y_test_clf, y_pred_best)*100:.2f}%")
print()
# Classification report shows precision, recall, F1 per class
# Precision = of all predicted Remote, how many were actually Remote?
# Recall    = of all actual Remote jobs, how many did we catch?
# F1-score  = harmonic mean of precision and recall
print("📊 Classification Report:")
print(classification_report(y_test_clf, y_pred_best, target_names=le_wm.classes_))


In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────
# Rows = actual labels, Columns = predicted labels
# Diagonal = correct predictions, off-diagonal = mistakes
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test_clf, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=le_wm.classes_,
            yticklabels=le_wm.classes_,
            linewidths=0.5)
ax.set_title(f'Confusion Matrix — {best_clf_name}', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('09_confusion_matrix.png', bbox_inches='tight')
plt.show()

total = cm.sum()
correct = cm.diagonal().sum()
print(f"Overall: {correct:,}/{total:,} correct ({correct/total*100:.1f}%)")
print(f"\n📖 Reading the matrix:")
print("  - Each ROW is the actual class")
print("  - Each COLUMN is the predicted class")
print("  - Diagonal cells (top-left to bottom-right) = CORRECT predictions")


## 💰 Section 5 — ML Task 2: Salary Prediction (Regression)

**Goal:** Predict `salary_midpoint_lpa` for tech jobs

**Why this is harder than Task 1:**
- Only **2,768 rows** have disclosed salary (out of 23K total)
- Small dataset = lower R² is expected and honest

**Metrics explained:**
| Metric | Meaning | Good value |
|--------|---------|------------|
| **MAE** | Average error in LPA | Lower = better |
| **RMSE** | Like MAE but penalises big mistakes more | Lower = better |
| **R²** | 0 = random guessing, 1 = perfect prediction | Closer to 1 = better |


In [ ]:
# ── Features for salary prediction ───────────────────────────
FEATURES_REG = [
    'role_category', 'primary_city', 'experience_min_yrs',
    'experience_max_yrs', 'skills_count', 'company_rating',
    'company_size_bucket', 'skill_domain', 'work_mode',
    'is_senior', 'is_fresher_friendly',
]
TARGET_REG = 'salary_midpoint_lpa'

# Use ONLY rows where salary was disclosed
reg_df = df[df['salary_disclosed'] == True].copy()
reg_df = reg_df[reg_df[TARGET_REG].between(1, 80)]  # remove extreme outliers

reg_df['is_senior']           = reg_df['is_senior'].astype(int)
reg_df['is_fresher_friendly'] = reg_df['is_fresher_friendly'].astype(int)

print(f"✅ Regression dataset: {len(reg_df):,} rows (salary disclosed & ≤80 LPA)")
print(f"   Salary range: ₹{reg_df[TARGET_REG].min():.1f} – ₹{reg_df[TARGET_REG].max():.1f} LPA")
print(f"   Median      : ₹{reg_df[TARGET_REG].median():.1f} LPA")
print(f"   Mean        : ₹{reg_df[TARGET_REG].mean():.1f} LPA")


In [ ]:
num_features_reg = ['experience_min_yrs','experience_max_yrs','skills_count',
                    'company_rating','is_senior','is_fresher_friendly']
cat_features_reg = ['role_category','primary_city','company_size_bucket',
                    'skill_domain','work_mode']

preprocessor_reg = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ]), num_features_reg),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), cat_features_reg),
])

X_reg = reg_df[FEATURES_REG]
y_reg = reg_df[TARGET_REG]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
print(f"✅ Train: {len(X_train_r):,} | Test: {len(X_test_r):,}")


In [ ]:
# ── Train & Compare Regression Models ────────────────────────
models_reg = {
    'Ridge Regression'  : Ridge(alpha=1.0),
    'Random Forest'     : RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=200, random_state=42,
                                                     learning_rate=0.05, max_depth=5),
}
if XGBOOST:
    models_reg['XGBoost'] = XGBRegressor(n_estimators=200, random_state=42,
                                          verbosity=0, n_jobs=-1,
                                          learning_rate=0.05, max_depth=6,
                                          subsample=0.8, colsample_bytree=0.8)

results_reg = []
print("Training regression models...\n")

for name, model in models_reg.items():
    pipe = Pipeline([('prep', preprocessor_reg), ('model', model)])
    pipe.fit(X_train_r, y_train_r)
    y_pred = pipe.predict(X_test_r)

    mae  = mean_absolute_error(y_test_r, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred))  # compatible all sklearn versions
    r2   = r2_score(y_test_r, y_pred)

    print(f"  {name:<20}  MAE=₹{mae:.2f} LPA  RMSE={rmse:.2f}  R²={r2:.4f}")
    results_reg.append({'Model': name, 'MAE (LPA)': mae, 'RMSE': rmse, 'R²': r2})

reg_results_df = pd.DataFrame(results_reg).sort_values('R²', ascending=False)
print(f"\n🏆 Best: {reg_results_df.iloc[0]['Model']}")
print(f"   R²  : {reg_results_df.iloc[0]['R²']:.4f}")
print(f"   MAE : ₹{reg_results_df.iloc[0]['MAE (LPA)']:.2f} LPA average prediction error")


In [ ]:
# ── Actual vs Predicted Plot ──────────────────────────────────
best_reg_name  = reg_results_df.iloc[0]['Model']
best_reg_pipe  = Pipeline([('prep', preprocessor_reg), ('model', models_reg[best_reg_name])])
best_reg_pipe.fit(X_train_r, y_train_r)
y_pred_reg = best_reg_pipe.predict(X_test_r)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted — perfect model = all points on red dashed line
axes[0].scatter(y_test_r, y_pred_reg, alpha=0.4, s=25, color='#2E86AB')
min_val = min(y_test_r.min(), y_pred_reg.min())
max_val = max(y_test_r.max(), y_pred_reg.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2,
             label='Perfect prediction line')
axes[0].set_title(f'Actual vs Predicted — {best_reg_name}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual Salary (LPA)')
axes[0].set_ylabel('Predicted Salary (LPA)')
axes[0].legend()

# Residuals — should be centred around 0 (no systematic bias)
residuals = y_test_r.values - y_pred_reg
axes[1].hist(residuals, bins=40, color='#A23B72', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--', lw=2, label='Zero error')
axes[1].set_title('Residuals (Actual − Predicted)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Residual (LPA)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('💰 Salary Prediction Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('10_salary_prediction.png', bbox_inches='tight')
plt.show()

r2_final  = r2_score(y_test_r, y_pred_reg)
mae_final = mean_absolute_error(y_test_r, y_pred_reg)
print(f"\n📊 Final — {best_reg_name}:")
print(f"   R² Score : {r2_final:.4f}  (model explains {r2_final*100:.1f}% of salary variance)")
print(f"   MAE      : ₹{mae_final:.2f} LPA  (average error)")


## 🔍 Section 6 — Feature Importance

Feature importance answers: **"Which inputs does the model rely on most?"**

A high importance score means that feature has a big impact on predictions.
Think of it as: *"Which clue is most helpful for solving the puzzle?"*


In [ ]:
# Feature importance — Work Mode Classifier
try:
    ohe_cats = (best_clf_pipe.named_steps['prep']
                             .named_transformers_['cat']
                             .named_steps['ohe']
                             .get_feature_names_out(cat_features_clf).tolist())
    all_features = num_features_clf + ohe_cats

    model_obj = best_clf_pipe.named_steps['model']
    if hasattr(model_obj, 'feature_importances_'):
        fi = pd.Series(model_obj.feature_importances_,
                       index=all_features).sort_values(ascending=False).head(15)

        fig, ax = plt.subplots(figsize=(10, 6))
        fi[::-1].plot(kind='barh', ax=ax, color='#2E86AB')
        ax.set_title(f'Top 15 Feature Importances — Work Mode ({best_clf_name})',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Importance Score')
        plt.tight_layout()
        plt.savefig('11_feature_importance_clf.png', bbox_inches='tight')
        plt.show()
        print("💡 Features at the TOP drive the model's decisions the most.")
        print("   Features near the BOTTOM have little influence.")
except Exception as e:
    print(f"Note: {e}")


In [ ]:
# Feature importance — Salary Predictor
try:
    ohe_cats_r = (best_reg_pipe.named_steps['prep']
                               .named_transformers_['cat']
                               .named_steps['ohe']
                               .get_feature_names_out(cat_features_reg).tolist())
    all_features_r = num_features_reg + ohe_cats_r

    model_obj_r = best_reg_pipe.named_steps['model']
    if hasattr(model_obj_r, 'feature_importances_'):
        fi_r = pd.Series(model_obj_r.feature_importances_,
                         index=all_features_r).sort_values(ascending=False).head(15)

        fig, ax = plt.subplots(figsize=(10, 6))
        fi_r[::-1].plot(kind='barh', ax=ax, color='#A23B72')
        ax.set_title(f'Top 15 Feature Importances — Salary ({best_reg_name})',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Importance Score')
        plt.tight_layout()
        plt.savefig('12_feature_importance_reg.png', bbox_inches='tight')
        plt.show()
except Exception as e:
    print(f"Note: {e}")


## ✅ Section 7 — Conclusion

### 📌 Key EDA Findings

| Finding | Detail |
|---------|--------|
| 🏙️ Top hiring city | Mumbai (14.1%) leads Bangalore (12.2%) |
| 💼 Most in-demand role | Data Scientist — 27.8% of all listings |
| 🏠 Work mode reality | 80.6% On-site — remote is still a minority in India |
| 💰 Median salary | ₹14.0 LPA (among disclosed listings only) |
| 🎓 Fresher opportunities | 18.0% of listings accept 0–1 year experience |
| 🔧 Top skill domain | Business Intelligence dominates (47.9%) |
| 📅 Job freshness | Most listings were posted 3+ weeks ago |

---

### 🤖 ML Model Results

| Task | Best Model | Score | Metric |
|------|-----------|-------|--------|
| ✅ Work Mode Classification | **XGBoost** | **90.2%** | Test Accuracy |
| ✅ Work Mode Classification | **XGBoost** | **90.1%** | CV-5fold Accuracy |
| 💰 Salary Prediction | **XGBoost** | **R² = 0.56** | 56% variance explained |
| 💰 Salary Prediction | **XGBoost** | **MAE = ₹4.48 LPA** | Avg prediction error |

> **Note on Salary R²:** 0.56 is honest — only 12% of listings disclose salary,
> giving us just ~2,700 training examples. With more salary data, R² would improve significantly.

---

### 💡 Business Insights for Job Seekers

1. **Mumbai vs Bangalore** — Mumbai leads in volume, but Bangalore likely leads in salary
2. **Remote jobs are rare** — only 9.3% are remote; filter specifically if you want WFH
3. **AI/ML skills pay more** — these roles earn 25–40% higher than average BI roles
4. **Experience = salary** — each extra year adds ~₹1.2 LPA on average
5. **Freshers: apply smart** — 4,182 listings explicitly welcome 0–1 year experience
6. **Negotiate your salary** — only 11.9% disclose salary; most are open to negotiation

---

### 🙏 If this notebook helped you, please **upvote ⬆️** — it keeps me motivated to create more!

---
*Notebook & Dataset by [Shreyash](https://www.kaggle.com/shree0910) | Indian Tech Jobs 2025 | Scraped from Naukri.com*
